In [2]:
import pandas as pd
import geopandas as gpd
import altair as alt
from pathlib import Path

In [56]:
MERGED_SF_TRACTS_SHP = (
    Path.cwd().parent / "clean-data/merged_sf_shapefiles/merged_sf_tracts.shp"
)

MERGED = Path.cwd().parent / "clean-data/merged_data.csv"

In [59]:
def create_tract_map(start_date: str, end_date: str, col_name: str):
    """
    Add docstring
    """
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    df = pd.read_csv(MERGED)
    sf_tracts = gpd.read_file(MERGED_SF_TRACTS_SHP)

    df["tract"] = df["tract"].astype(str).str.zfill(11)

    # filtered_df = (
    #     df[(df["date"] >= start_date) & (df["date"] <= end_date)]
    #     .groupby("tract")
    #     # .agg(metric=(col_name, "mean"))
    #     .agg(rent=("median_rent", "mean"),
    #          evictions=("eviction_rate", "mean"),
    #          estimate=("estimate", "mean"),
    #          reports=("311_calls","mean"),
    #          metric=(col_name, "mean"))
    #     .reset_index()
    # )

    # # Merge filtered, aggregated dataframe with GeoDataFrame
    # merged_gdf = sf_tracts.merge(filtered_df, left_on="GEOID", right_on="tract", how="left")


    # tooltips = [
    #     alt.Tooltip("GEOID:N", title="Tract ID"),
    #     alt.Tooltip("population:Q", title="Population"),
    #     alt.Tooltip("rent:Q", title="Median rent (per month)"),
    #     alt.Tooltip("evictions:Q", title="Average eviction rate"),
    #     alt.Tooltip("estimate:Q", title="Average homeless population estimate"),
    #     alt.Tooltip("reports:Q", title="Average citizen-reported encampments")
    # ]

    # # if col_name not in ["median_rent", "eviction_rate", "estimate", "311_calls"]:
    # #     tooltips.append(alt.Tooltip("metric:Q", title=f"Average {METRIC_NAMES[col_name].lower()}"))

    # # Build chart
    # chart = (
    #     alt.Chart(merged_gdf)
    #     .mark_geoshape()
    #     .encode(
    #         color=alt.Color("metric:Q", title={METRIC_NAMES[col_name]}),
    #         tooltip=tooltips
    #     )
    #     .transform_lookup(
    #         lookup="GEOID",
    #         from_=alt.LookupData(filtered_df, ("tract"), ["metric"]),
    #     )
    #     .project(type="albersUsa")
    #     .properties(
    #         title=f"Average {METRIC_NAMES[col_name].lower()} in SF tracts ({start_date} to {end_date})"
    #     )
    #     .interactive()
    #     )

    filtered_df = (
        df[(df["date"] >= start_date) & (df["date"] <= end_date)]
        .groupby("tract")
        .agg(metric=(col_name, "mean"))
        .reset_index()
    )

    filtered_df["tract"] = filtered_df["tract"].astype(str).str.zfill(11)

    chart = (
        alt.Chart(sf_tracts)
        .mark_geoshape(stroke="black", strokeWidth=0.5)
        .encode(
            color=alt.Color(
                "metric:Q",
                title=METRIC_NAMES[col_name],
                legend=alt.Legend(
                    orient="right",
                    direction="vertical",
                    labelAlign="left",
                    titlePadding=5,
                    offset=10,
                ),
            ),
            tooltip=[
                alt.Tooltip("GEOID:N", title="Tract ID"),
                alt.Tooltip("population:Q", title="Population"),
                alt.Tooltip("med_hh_inc:Q", title="Median annual household income"),
                alt.Tooltip("white_pct:Q", title="% white population"),
                alt.Tooltip(
                    "metric:Q",
                    title=f"Monthly average {METRIC_NAMES[col_name].lower()}",
                ),
            ],
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, "tract", ["metric"]),
        )
        .project(type="albersUsa")
        .interactive()
    )

    return filtered_df
    return chart


### CHANGE ROUNDING
### Include another layer for the Lincoln Park tract?
### Do we want monthly household income instead of annual?

In [69]:
def create_tract_map(source_file: Path, start_date: str, end_date: str, col_name: str):
    """
    Add docstring
    """
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    df = pd.read_csv(source_file)
    sf_tracts = gpd.read_file(MERGED_SF_TRACTS_SHP)
    ### TBD ON WHETHER THIS SHOULD BE INSIDE OR OUTSIDE FUNCTION

    df["tract"] = df["tract"].astype(str).str.zfill(11)

    filtered_df = (
        df[(df["date"] >= start_date) & (df["date"] <= end_date)]
        .groupby("tract")
        .agg(metric=(col_name, "mean"))
        .reset_index()
    )

    chart = (
        alt.Chart(sf_tracts)
        .mark_geoshape()
        .encode(
            color=alt.Color("metric:Q", title={METRIC_NAMES[col_name]}),
            tooltip=[
                alt.Tooltip("GEOID:N", title="Tract ID"),
                alt.Tooltip("population:Q", title="Population"),
                alt.Tooltip("med_hh_inc:Q", title="Median annual household income"),
                alt.Tooltip("white_pct:Q", title="% white population"),
                alt.Tooltip(
                    "metric:Q",
                    title=f"Monthly average {METRIC_NAMES[col_name].lower()}",
                ),
            ],
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, ("tract"), ["metric"]),
        )
        .project(type="albersUsa")
        .properties(
            title=f"Average {METRIC_NAMES[col_name].lower()} in SF tracts ({start_date} to {end_date})"
        )
        .interactive()
    )

    return df

In [70]:
create_tract_map(MERGED, "2020-01", "2024-12", "estimate")

,date,tract,median_rent,eviction_rate,311_calls,tents,structures,vehicles,estimate
0,2024-03,6075016200.0,2606.585322,0.001801,20.0,1.6,0.4,0.0,3.72
1,2024-03,6075016801.0,3015.443906,0.000000,7.0,0.5,0.5,0.5,2.60
2,2024-03,6075012100.0,2620.810346,0.000558,6.0,0.0,0.0,0.0,0.00
3,2024-03,6075061501.0,3610.901625,0.000000,16.0,2.5,0.5,0.0,5.60
4,2024-03,6075016102.0,2953.636158,0.000870,1.0,0.0,0.0,3.8,6.08
...,...,...,...,...,...,...,...,...,...
13269,2024-11,6075010102.0,3228.442380,0.000000,3.0,0.0,0.0,0.0,0.00
13270,2024-11,6075010600.0,3228.442380,0.000640,8.0,0.0,0.0,0.0,0.00
13271,>>>>>>> aaf4c33bc3717cf6ac85dab58e58bb12df2ff2e0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13272,2024-11,6075010401.0,3228.442380,0.000000,2.0,0.0,0.0,0.0,0.00
